# Step 04C — Audit + final statistics (NO model inference)

**Scope:** Read the saved **20/20** unified COCO evaluation outputs from the repaired run, audit the **13 zero-area/clipped predictions**, and produce a transparent statistical evidence bundle.  
**Does NOT** train, perform inference, load model weights, or alter source data. **No GPU needed.**  

The earlier stop was an intentionally conservative quality-control threshold (**more than 10 invalid boxes**), not a TorchMetrics/CUDA failure. The audit below verifies geometry, preserves details, and treats the harmonized AP as a **secondary analysis subject to the documented prediction-filtering limitation**.

**Run all cells.** Upload the resulting ZIP to ChatGPT.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime,timezone
from itertools import combinations
import numpy as np, pandas as pd, zipfile, json, hashlib, shutil
from scipy.stats import friedmanchisquare,wilcoxon

ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
REV=ROOT/'reviewer_revision'
SOURCE=REV/'Step_04C_Repair_Resume_20261024T180342Z'
assert SOURCE.is_dir(),f'Cannot find the previous finished 20/20 run: {SOURCE}'
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REV/f'Step_04C_QC_Audit_Finish_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
FAMILIES=['YOLO11s','RT-DETR-L','Faster R-CNN','RetinaNet']
print('Reading EXISTING run:',SOURCE)
print('Saving review bundle:',OUT)
print('No models or GPUs are used.')

In [ ]:
# 1. Confirm all required previously generated evidence exists and is internally consistent.
metrics=pd.read_csv(SOURCE/'unified_coco_per_fold.csv')
assert len(metrics)==20 and len(metrics[['family','fold']].drop_duplicates())==20,'20 distinct folds required'
assert (metrics.status=='OK').all(),'The existing repair run has errors; DO NOT finalize.'
assert set(metrics.family)==set(FAMILIES)
assert all(set(metrics[metrics.family==f].fold)==set(range(1,6)) for f in FAMILIES)
assert (metrics.n_images==14).all(),'Some folds do not have 14 images'
assert list(metrics.sort_values(['family','fold']).groupby('family').n_gt.sum().values)==[206]*4,'GT count differs by model'
assert all((SOURCE/f'predictions_{f.replace(" ","_")}_fold{k}.csv').is_file() for f in FAMILIES for k in range(1,6))
assert (SOURCE/'checkpoint_sha256_verified.csv').is_file()
assert (SOURCE/'fold_gt_manifest_sha256.csv').is_file()
ck=pd.read_csv(SOURCE/'checkpoint_sha256_verified.csv')
gtman=pd.read_csv(SOURCE/'fold_gt_manifest_sha256.csv')
assert len(ck)==20 and ck.sha256.str.fullmatch('[0-9a-f]{64}').all()
assert len(gtman)==70 and gtman.image_sha256.str.fullmatch('[0-9a-f]{64}').all()
for _,r in metrics.iterrows():
    name=f'predictions_{r["family"].replace(" ","_")}_fold{int(r["fold"])}.csv'
    t=pd.read_csv(SOURCE/name)
    assert (t.prediction_index.ge(0).sum())==int(r.n_predictions),(name,'count mismatch')
print('PASS: 20/20 completed folds, 20 checkpoint records, 70 reference image-label records, 20 prediction exports.')
print(metrics[['family','fold','COCO_mAP50','COCO_mAP50_95','n_predictions']].to_string(index=False))

In [ ]:
# 2. Independently inspect ALL 13 omitted boxes, before accepting any result.
a=pd.read_csv(SOURCE/'dropped_invalid_prediction_boxes.csv')
required={'family','fold','image','confidence','x1_raw','y1_raw','x2_raw','y2_raw','x1_clipped','y1_clipped','x2_clipped','y2_clipped','reason'}
assert required.issubset(a.columns),required-set(a.columns)
assert len(a)==13,f'Expected the disclosed 13 items; got {len(a)}. Manually review instead.'
assert a.family.eq('YOLO11s').all() and a.fold.eq(1).all()
assert a.image.nunique()==13, 'Unexpected distribution; stop and investigate.'
floatcols=['confidence','x1_raw','y1_raw','x2_raw','y2_raw','x1_clipped','y1_clipped','x2_clipped','y2_clipped']
assert np.isfinite(a[floatcols].to_numpy(float)).all(),'Non-finite values are a separate bug.'
assert a.confidence.ge(.001-1e-7).all(),'Invalid logged threshold/confidence'
a=a.merge(gtman.loc[gtman.fold==1,['image','W','H','n_gt']],on='image',how='left',validate='many_to_one')
assert a[['W','H']].notna().all().all(),'Unknown image dimensions'
a['raw_valid_xyxy']=(a.x2_raw>a.x1_raw)&(a.y2_raw>a.y1_raw)
a['clipped_valid_xyxy']=(a.x2_clipped>a.x1_clipped)&(a.y2_clipped>a.y1_clipped)
a['fully_outside_image']=(a.x1_raw>=a.W)|(a.x2_raw<=0)|(a.y1_raw>=a.H)|(a.y2_raw<=0)
a['clipped_zero_width']=a.x2_clipped<=a.x1_clipped
a['clipped_zero_height']=a.y2_clipped<=a.y1_clipped
a['expected_x1']=a.x1_raw.clip(lower=0)
a['expected_y1']=a.y1_raw.clip(lower=0)
a['expected_x2']=a.x2_raw.clip(lower=0)
a['expected_y2']=a.y2_raw.clip(lower=0)
a['expected_x1']=np.minimum(a.expected_x1,a.W)
a['expected_x2']=np.minimum(a.expected_x2,a.W)
a['expected_y1']=np.minimum(a.expected_y1,a.H)
a['expected_y2']=np.minimum(a.expected_y2,a.H)
for axis in ['x1','y1','x2','y2']:
    assert np.allclose(a[f'{axis}_clipped'],a[f'expected_{axis}'],atol=1e-4),f'Clipping values disagree: {axis}'
assert (~a.clipped_valid_xyxy).all(),'An allegedly discarded box was actually valid'
flags={'invalid_total':len(a),'raw_degenerate_count':int((~a.raw_valid_xyxy).sum()),
       'positive_raw_extent_count':int(a.raw_valid_xyxy.sum()),
       'fully_outside_image_count':int(a.fully_outside_image.sum()),
       'zero_width_after_clip_count':int(a.clipped_zero_width.sum()),
       'zero_height_after_clip_count':int(a.clipped_zero_height.sum()),
       'score_min':float(a.confidence.min()),'score_median':float(a.confidence.median()),
       'score_max':float(a.confidence.max()),'affected_image_count':int(a.image.nunique())}
print('GEOMETRY AUDIT:')
print(json.dumps(flags,indent=2))
print(a[['image','confidence','x1_raw','y1_raw','x2_raw','y2_raw','W','H','reason','raw_valid_xyxy','fully_outside_image']].to_string(index=False))
a.drop(columns=[x for x in a.columns if x.startswith('expected_')]).to_csv(OUT/'invalid_box_geometry_audit.csv',index=False)
(OUT/'invalid_box_audit_counts.json').write_text(json.dumps(flags,indent=2))
# Stop if filtering could be based on something other than invalid model geometry.
assert len(a)==13 and (~a.clipped_valid_xyxy).all()
print('PASS: each of the 13 omissions is grounded in non-positive POST-CLIP geometry.')
print('CAUTION: passing this check does NOT prove that omitting the predictions has no influence on AP.')

In [ ]:
# 3. Reconstruct secondary statistics from exactly the saved unified-COCO AP outputs.
summary=[];testrows=[];pairrows=[]
for m in ['COCO_mAP50','COCO_mAP50_95']:
    wide=metrics.pivot(index='fold',columns='family',values=m)[FAMILIES]
    assert wide.shape==(5,4) and not wide.isna().any().any()
    for f in FAMILIES:
        y=wide[f]
        summary.append(dict(model=f,metric=m,n_folds=5,mean=float(y.mean()),sample_SD=float(y.std(ddof=1)),min=float(y.min()),max=float(y.max())))
    fr=friedmanchisquare(*(wide[f] for f in FAMILIES))
    testrows.append(dict(metric=m,friedman_chi2=float(fr.statistic),friedman_p_asymptotic=float(fr.pvalue),n_folds=5))
    local=[]
    for u,v in combinations(FAMILIES,2):
        w=wilcoxon(wide[u],wide[v],alternative='two-sided',zero_method='wilcox',method='auto')
        local.append(dict(metric=m,model_a=u,model_b=v,W=float(w.statistic),p_raw=float(w.pvalue)))
    order=sorted(range(6),key=lambda i:local[i]['p_raw']);prev=0
    for rank,j in enumerate(order):
        prev=max(prev,min(1.0,(6-rank)*local[j]['p_raw']))
        local[j]['p_Holm']=prev;local[j]['significant_at_0_05']=bool(prev<.05)
    pairrows.extend(local)
summ=pd.DataFrame(summary);frit=pd.DataFrame(testrows);pair=pd.DataFrame(pairrows)
summ.to_csv(OUT/'unified_COCO_model_mean_sd.csv',index=False)
frit.to_csv(OUT/'unified_COCO_friedman.csv',index=False)
pair.to_csv(OUT/'unified_COCO_wilcoxon_holm.csv',index=False)
print('UNIFIED COCO — MODEL SUMMARY')
print(summ.to_string(index=False,float_format=lambda v:f'{v:.6f}'))
print('FRIEDMAN')
print(frit.to_string(index=False,float_format=lambda v:f'{v:.6f}'))
print('PAIRWISE SIGNIFICANT AFTER HOLM:',int(pair.significant_at_0_05.sum()))
# All statistics are exploratory (five overlapping training folds).
# The zero-area filter is recorded as a limitation; no architecture-only claim is supported.

In [ ]:
# 4. Assemble a traceable audit package without touching any prior source files.
for name in ['unified_coco_per_fold.csv','checkpoint_sha256_verified.csv','fold_gt_manifest_sha256.csv',
             'run_environment_and_protocol.json','SOURCE_RUN_REFERENCE.txt','dropped_invalid_prediction_boxes.csv']:
    p=SOURCE/name
    if p.is_file():shutil.copy2(p,OUT/name)
for f in FAMILIES:
  for k in range(1,6):
    name=f'predictions_{f.replace(" ","_")}_fold{k}.csv'
    shutil.copy2(SOURCE/name,OUT/name)
notes=f"""STEP 04C — POST-REPAIR QC + STATISTICAL SUMMARY (WORKING EVIDENCE)

Original repaired run: {SOURCE}
Evaluation: 20/20 checkpoints, 5 folds x 4 pipelines, same validation folds.
No new model inference or training was performed in this notebook.
One model/fold (YOLO11s fold 1) originally failed because a predicted box had non-positive clipped area.
The repair resumed this fold and excluded 13 geometrically invalid post-clipping boxes on 13 images.
Original coordinates and confidence scores for all 13 are preserved in invalid_box_geometry_audit.csv.
The earlier >10 threshold was a QA tripwire, NOT a hypothesis test or a principled cutoff.
The effect of omitting these detections on AP has NOT been fully quantified; do not claim exact parity with all original native evaluators.
These unified COCO metrics are a SECONDARY harmonized evaluation sensitivity/control analysis.
They do not harmonize preprocessing, training budgets, checkpoint-selection, or hyperparameters.
The original source annotations have known limitations from independent expert review.
Five overlapping CV training folds limit inferential power.
DO NOT directly replace manuscript comparative figures without auditing full data provenance.
"""
(OUT/'README_AND_LIMITATIONS.txt').write_text(notes)
zip_path=OUT/'Step_04C_QC_Audit_Finish_Results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
  for p in sorted(OUT.iterdir()):
    if p.is_file() and p!=zip_path:z.write(p,p.name)
print('SUCCESS: all 20 folds available; geometry audit and statistics saved.')
print('ZIP:',zip_path)
print('Size:',zip_path.stat().st_size,'bytes')
print('Upload this ZIP to ChatGPT for a final methodological audit.')